# Features experiment

In [15]:
import polars as pl
import numpy as np
import mlflow
import mlflow.catboost
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from catboost import Pool, CatBoostClassifier
import os

# Целевые переменные (маленькие, можно загрузить сразу)
target = pl.read_parquet('data/train_target.parquet')
sample_submit = pl.read_parquet('data/sample_submit.parquet')

In [16]:
FEATURES = 'extra'               # 'main', 'extra' или 'both'
REMOVE_CONSTANT = False
REMOVE_MISSING = False
REMOVE_CORRELATED = False
REMOVE_LEAKAGE = True 

EXPERIMENT_NAME = f"exp_{FEATURES}_const{REMOVE_CONSTANT}_miss{REMOVE_MISSING}_corr{REMOVE_CORRELATED}_leak{REMOVE_LEAKAGE}"

In [17]:
if FEATURES == 'main':
    train_lazy = pl.scan_parquet('data/train_main_features.parquet')
    test_lazy = pl.scan_parquet('data/test_main_features.parquet')
elif FEATURES == 'extra':
    train_lazy = pl.scan_parquet('data/train_extra_features.parquet')
    test_lazy = pl.scan_parquet('data/test_extra_features.parquet')
elif FEATURES == 'both':
    train_main = pl.scan_parquet('data/train_main_features.parquet')
    train_extra = pl.scan_parquet('data/train_extra_features.parquet')
    test_main = pl.scan_parquet('data/test_main_features.parquet')
    test_extra = pl.scan_parquet('data/test_extra_features.parquet')
    train_lazy = train_main.join(train_extra, on='customer_id', how='inner')
    test_lazy = test_main.join(test_extra, on='customer_id', how='inner')
else:
    raise ValueError("FEATURES должен быть 'main', 'extra' или 'both'")

In [18]:
train_df = train_lazy.collect()
test_df = test_lazy.collect()
print(f"Размер train: {train_df.shape}, test: {test_df.shape}")

: 

In [14]:
# Удаление константных
if REMOVE_CONSTANT:
    unique_counts = train_lazy.select(pl.all().n_unique()).collect().row(0)
    const_cols = [col for col, cnt in zip(train_lazy.columns, unique_counts) if cnt == 1]
    train_lazy = train_lazy.drop(const_cols)
    test_lazy = test_lazy.drop([c for c in const_cols if c in test_lazy.columns])
    print(f"Удалено константных колонок: {len(const_cols)}")

# Удаление колонок с пропусками
if REMOVE_MISSING:
    null_counts = train_lazy.select(pl.all().null_count()).collect().row(0)
    missing_cols = [col for col, cnt in zip(train_lazy.columns, null_counts) if cnt > 0]
    train_lazy = train_lazy.drop(missing_cols)
    test_lazy = test_lazy.drop([c for c in missing_cols if c in test_lazy.columns])
    print(f"Удалено колонок с пропусками: {len(missing_cols)}")

# Удаление коррелированных (считаем на всех данных)
if REMOVE_CORRELATED:
    print("Загружаем данные для расчёта корреляций...")
    df_corr = train_lazy.collect()
    corr_matrix = df_corr.drop('customer_id').to_pandas().corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > 0.95)]
    train_lazy = train_lazy.drop(to_drop)
    test_lazy = test_lazy.drop([c for c in to_drop if c in test_lazy.columns])
    print(f"Удалено коррелированных колонок: {len(to_drop)}")

# Удаление утечек (считаем на всех данных)
if REMOVE_LEAKAGE:
    print("Загружаем данные для выявления утечек...")
    df_leak = train_lazy.collect()
    combined = df_leak.join(target, on='customer_id', how='inner')
    pdf = combined.to_pandas()
    feature_cols = df_leak.columns
    target_cols = target.drop('customer_id').columns
    corr_with_target = pdf[feature_cols].corrwith(pdf[target_cols])
    leak_cols = [col for col in feature_cols if np.abs(corr_with_target[col]).max() > 0.9]
    train_lazy = train_lazy.drop(leak_cols)
    test_lazy = test_lazy.drop([c for c in leak_cols if c in test_lazy.columns])
    print(f"Удалено потенциально утекающих колонок: {len(leak_cols)}")

Загружаем данные для выявления утечек...
Удалено потенциально утекающих колонок: 0


In [ ]:
# Отделяем customer_id и таргеты
X = train_df.drop('customer_id')
y = target.drop('customer_id')

# Сплит
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=None
)

# Категориальные признаки
cat_feature_names = [col for col in X_train.columns if col.startswith("cat_feature")]
X_train = X_train.with_columns(pl.col(cat_feature_names).cast(pl.Int32))
X_val = X_val.with_columns(pl.col(cat_feature_names).cast(pl.Int32))

# Конвертация в pandas
X_train_pd = X_train.to_pandas()
X_val_pd = X_val.to_pandas()
y_train_pd = y_train.to_pandas()
y_val_pd = y_val.to_pandas()

# MLflow run
mlflow.set_experiment(EXPERIMENT_NAME)
with mlflow.start_run():
    mlflow.log_param("features", FEATURES)
    mlflow.log_param("remove_constant", REMOVE_CONSTANT)
    mlflow.log_param("remove_missing", REMOVE_MISSING)
    mlflow.log_param("remove_correlated", REMOVE_CORRELATED)
    mlflow.log_param("remove_leakage", REMOVE_LEAKAGE)
    mlflow.log_param("n_features", X_train.shape[1])
    mlflow.log_param("train_size", len(X_train))
    mlflow.log_param("val_size", len(X_val))


    model = CatBoostClassifier(iterations = 10, 
                           depth = 4, 
                           learning_rate = 0.25, 
                           loss_function = 'MultiLogloss', 
                           nan_mode = 'Min', 
                           random_seed = 1234,
                           task_type='GPU',        # ← добавляем эту строчку
                           devices='0', 
                           verbose = 1)

    train_pool = Pool(X_train_pd, label=y_train_pd, cat_features=cat_feature_names)
    val_pool = Pool(X_val_pd, label=y_val_pd, cat_features=cat_feature_names)
    model.fit(train_pool)

    # Оценка на валидации
    val_preds = model.predict_proba(val_pool)
    target_cols = y.columns
    auc_scores = []
    for i, col in enumerate(target_cols):
        if y_val_pd[col].nunique() == 2:
            auc = roc_auc_score(y_val_pd[col], val_preds[:, i])
            auc_scores.append(auc)
            mlflow.log_metric(f"val_auc_{col}", auc)
    macro_auc = np.mean(auc_scores)
    mlflow.log_metric("val_macro_auc", macro_auc)
    print(f"Macro AUC: {macro_auc:.4f}")

    # Сохраняем модель
    mlflow.catboost.log_model(model, "catboost_model")

    # Сабмит
    test_df = test_df.with_columns(pl.col(cat_feature_names).cast(pl.Int32))
    test_pool = Pool(test_df.drop('customer_id').to_pandas(), cat_features=cat_feature_names)
    test_preds = model.predict_proba(test_pool)
    predict_schema = [col.replace("target_", "predict_") for col in target_cols]
    submit = test_df.select('customer_id').hstack(pl.DataFrame(test_preds, schema=predict_schema))
    submit_path = f"submissions/{EXPERIMENT_NAME}.parquet"
    os.makedirs("submissions", exist_ok=True)
    submit.write_parquet(submit_path)
    mlflow.log_artifact(submit_path)
    print(f"Сабмит сохранён: {submit_path}")

2026/03/20 21:14:17 INFO mlflow.tracking.fluent: Experiment with name 'exp_main_constFalse_missFalse_corrFalse_leakFalse' does not exist. Creating a new experiment.


0:	learn: 0.2894499	total: 806ms	remaining: 7.26s
1:	learn: 0.1694639	total: 1.56s	remaining: 6.25s
2:	learn: 0.1271867	total: 2.33s	remaining: 5.43s
3:	learn: 0.1097094	total: 3.09s	remaining: 4.64s
4:	learn: 0.1017718	total: 3.86s	remaining: 3.86s
5:	learn: 0.0975273	total: 4.6s	remaining: 3.06s
6:	learn: 0.0952471	total: 5.16s	remaining: 2.21s
7:	learn: 0.0938029	total: 5.81s	remaining: 1.45s
8:	learn: 0.0930777	total: 6.57s	remaining: 730ms
9:	learn: 0.0924268	total: 7.17s	remaining: 0us
Macro AUC: 0.7449
Сабмит сохранён: submissions/exp_main_constFalse_missFalse_corrFalse_leakFalse.parquet
